In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
!pip install mlflow dagshub -q
import os
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.feature_selection import SelectFromModel, RFE
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

os.environ["MLFLOW_TRACKING_USERNAME"] = "giomamaca"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "095241f97fd50006565a872843e1054bb2ae82cb"

DAGSHUB_USER = 'giomamaca'
DAGSHUB_REPO = 'ML2'
mlflow.set_tracking_uri(f'https://dagshub.com/{DAGSHUB_USER}/{DAGSHUB_REPO}.mlflow')

EXPERIMENT_NAME = 'AdaBoost_Training'
mlflow.set_experiment(EXPERIMENT_NAME)

SEED = 42
np.random.seed(SEED)

In [3]:
def run_exists(run_name, experiment_name='AdaBoost_Training'):
    client = MlflowClient()
    exp    = client.get_experiment_by_name(experiment_name)
    if exp is None:
        return False
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=f"tags.mlflow.runName = '{run_name}'"
    )
    return len(runs) > 0

def get_run(run_name, experiment_name='AdaBoost_Training'):
    client = MlflowClient()
    exp    = client.get_experiment_by_name(experiment_name)
    runs   = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=f"tags.mlflow.runName = '{run_name}'"
    )
    return runs[0]

def delete_run(run_name, experiment_name='AdaBoost_Training'):
    client = MlflowClient()
    exp    = client.get_experiment_by_name(experiment_name)
    runs   = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=f"tags.mlflow.runName = '{run_name}'"
    )
    for run in runs:
        client.delete_run(run.info.run_id)
        print(f"Deleted: {run_name} (id={run.info.run_id})")

def run_experiment(run_name, params, X_tr, y_tr, X_val, y_val):
    if run_exists(run_name):
        print(f'{run_name} already logged, skipping...')
        return
    with mlflow.start_run(run_name=run_name):
        base_max_depth = params.pop('base_max_depth')
        est = DecisionTreeClassifier(max_depth=base_max_depth, random_state=SEED)
        m   = AdaBoostClassifier(estimator=est, **params)
        m.fit(X_tr, y_tr)
        train_auc = roc_auc_score(y_tr, m.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
        gap       = train_auc - val_auc
        state     = 'underfitting' if val_auc < 0.80 and gap < 0.02 else 'overfitting' if gap > 0.05 else 'good'
        print(f'{run_name}: Train={train_auc:.5f}  Val={val_auc:.5f}  → {state}')
        mlflow.log_params({**params, 'base_max_depth': base_max_depth})
        mlflow.log_metrics({'train_auc': train_auc, 'val_auc': val_auc, 'gap': gap})
        mlflow.log_param('model_state', state)

In [ ]:
train_trans  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_id     = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_trans   = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_id      = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

train = train_trans.merge(train_id, on='TransactionID', how='left')
test  = test_trans.merge(test_id,  on='TransactionID', how='left')

print('Train shape:', train.shape)
print('Test shape :', test.shape)
print('Fraud rate  :', train['isFraud'].mean().round(4))

## Preprocessing

In [ ]:
class FraudPreprocessor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = X.copy()

        nan_frac        = X.isnull().mean()
        self.nan_drop_  = nan_frac[nan_frac > 0.90].index.tolist()
        num_cols        = [c for c in X.select_dtypes('number').columns
                           if c not in ['TransactionID', 'isFraud']]
        self.near_zero_ = X[num_cols].std()[X[num_cols].std() < 0.01].index.tolist()
        X.drop(columns=self.nan_drop_ + self.near_zero_, inplace=True, errors='ignore')

        X = self._add_features(X)

        self.freq_maps_ = {}
        for col in ['card1', 'card2', 'addr1', 'P_emaildomain']:
            if col in X.columns:
                self.freq_maps_[col] = X[col].value_counts(normalize=True).to_dict()

        self.grp_stats_ = {}
        for grp in ['card1', 'card2']:
            if grp in X.columns:
                stats = X.groupby(grp)['TransactionAmt'].agg(['mean', 'std', 'count'])
                stats.columns = [f'{grp}_amt_{s}' for s in ['mean', 'std', 'count']]
                self.grp_stats_[grp] = stats

        if 'card1' in X.columns and 'card2' in X.columns:
            combined = X['card1'].astype(str) + '_' + X['card2'].astype(str)
            self.card_freq_ = combined.value_counts(normalize=True).to_dict()

        self.encoders_ = {}
        for col in X.select_dtypes('object').columns:
            if col not in ['TransactionID']:
                le = LabelEncoder()
                le.fit(X[col].astype(str).fillna('unknown'))
                self.encoders_[col] = le

        print(f'Label encoders fit on {len(self.encoders_)} columns: {list(self.encoders_.keys())}')

        self.feature_names_ = self._transform(X.copy()).columns.tolist()
        return self

    def _add_features(self, X):
        if 'id_30' in X.columns:
            X['os'] = X['id_30'].apply(
                lambda v: next((o for o in ['android','ios','windows','mac','linux']
                                if o in str(v).lower()), 'other') if pd.notna(v) else 'unknown')
        if 'id_31' in X.columns:
            X['browser'] = X['id_31'].apply(
                lambda v: next((b for b in ['chrome','safari','firefox','edge','samsung','opera']
                                if b in str(v).lower()), 'other') if pd.notna(v) else 'unknown')

        if 'TransactionDT' in X.columns:
            X['transaction_day']     = (X['TransactionDT'] // 86400).astype('int32')
            X['transaction_hour']    = ((X['TransactionDT'] // 3600) % 24).astype('int32')
            X['transaction_weekday'] = (X['transaction_day'] % 7).astype('int32')
            X['transaction_month']   = (X['transaction_day'] // 30).astype('int32')
            X['is_night']            = X['transaction_hour'].between(0, 6).astype('int8')
            X['is_weekend']          = X['transaction_weekday'].isin([5, 6]).astype('int8')

        X['TransactionAmt_log']   = np.log1p(X['TransactionAmt'])
        X['TransactionAmt_cents'] = (X['TransactionAmt'] * 100 % 100).astype('float32')
        X['amt_is_round']         = (X['TransactionAmt_cents'] == 0).astype('int8')

        if 'P_emaildomain' in X.columns:
            X['P_email_suffix'] = X['P_emaildomain'].apply(
                lambda x: str(x).split('.')[-1] if pd.notna(x) else 'unknown')
        if 'R_emaildomain' in X.columns:
            X['R_email_suffix'] = X['R_emaildomain'].apply(
                lambda x: str(x).split('.')[-1] if pd.notna(x) else 'unknown')
        if 'P_emaildomain' in X.columns and 'R_emaildomain' in X.columns:
            X['email_match'] = (X['P_emaildomain'] == X['R_emaildomain']).astype('int8')

        return X

    def _transform(self, X):
        for col, fmap in getattr(self, 'freq_maps_', {}).items():
            if col in X.columns:
                X[f'{col}_freq'] = X[col].map(fmap).fillna(0).astype('float32')

        for grp, stats in getattr(self, 'grp_stats_', {}).items():
            if grp in X.columns:
                X = X.merge(stats, on=grp, how='left')

        if hasattr(self, 'card_freq_') and 'card1' in X.columns and 'card2' in X.columns:
            combined = X['card1'].astype(str) + '_' + X['card2'].astype(str)
            X['card1_card2_freq'] = combined.map(self.card_freq_).fillna(0).astype('float32')

        for col, le in getattr(self, 'encoders_', {}).items():
            if col in X.columns:
                X[col] = X[col].astype(str)
                known  = set(le.classes_)
                X[col] = X[col].apply(lambda v: v if v in known else le.classes_[0])
                X[col] = le.transform(X[col])

        drop = [c for c in ['TransactionID', 'TransactionDT', 'isFraud'] if c in X.columns]
        X.drop(columns=drop, inplace=True)

        return X.fillna(-999)

    def transform(self, X):
        X = X.copy()
        drop = [c for c in self.nan_drop_ + self.near_zero_ if c in X.columns]
        X.drop(columns=drop, inplace=True)
        for col in X.select_dtypes('float64').columns:
            X[col] = X[col].astype('float32')
        for col in X.select_dtypes('int64').columns:
            X[col] = X[col].astype('int32')
        X = self._add_features(X)
        X = self._transform(X)

        for col in self.feature_names_:
            if col not in X.columns:
                X[col] = -999
        return X[self.feature_names_].fillna(-999)

## Cleaning + Feature Engineering

In [ ]:
preprocessor = FraudPreprocessor()

if not run_exists('AdaBoost_Cleaning'):
    with mlflow.start_run(run_name='AdaBoost_Cleaning'):
        preprocessor.fit(train.drop(columns=['isFraud']))
        mlflow.log_params({
            'dropped_high_nan' : len(preprocessor.nan_drop_),
            'dropped_near_zero': len(preprocessor.near_zero_),
            'remaining_cols'   : len(preprocessor.feature_names_)
        })
else:
    print('AdaBoost_Cleaning already logged, skipping...')
    preprocessor.fit(train.drop(columns=['isFraud']))

if not run_exists('AdaBoost_Feature_Engineering'):
    with mlflow.start_run(run_name='AdaBoost_Feature_Engineering'):
        mlflow.log_metric('total_features', len(preprocessor.feature_names_))
else:
    print('AdaBoost_Feature_Engineering already logged, skipping...')

X      = preprocessor.transform(train.drop(columns=['isFraud']))
X_test = preprocessor.transform(test)
y      = train['isFraud']

print('X shape     :', X.shape)
print('X_test shape:', X_test.shape)

## Feature Selection

In [ ]:
TARGET    = 'isFraud'
DROP_COLS = ['TransactionID', 'TransactionDT', TARGET]

X = preprocessor.transform(train.drop(columns=['isFraud']))
y = train[TARGET]

for col in X.select_dtypes('object').columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str).fillna('unknown'))

print(X.select_dtypes('object').columns.tolist())
print(X.shape)

In [ ]:
SKF_FAST = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
SKF      = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# AdaBoost base estimator
BASE_EST    = DecisionTreeClassifier(max_depth=2, random_state=SEED)
BASE_PARAMS = dict(estimator=BASE_EST, n_estimators=100, random_state=SEED)

sample_idx = X.sample(frac=0.3, random_state=SEED).index
X_sample   = X.loc[sample_idx]
y_sample   = y.loc[sample_idx]

def _compute_all_feature_sets(X, X_sample, y_sample):
    num_X     = X.select_dtypes(include='number')
    sample    = num_X.sample(frac=0.2, random_state=SEED)
    corr      = sample.corr().abs()
    upper     = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop_corr = [col for col in upper.columns if any(upper[col] > 0.95)]
    feats_A   = [c for c in X.columns if c not in drop_corr]

    ab_imp      = AdaBoostClassifier(**BASE_PARAMS)
    ab_imp.fit(X_sample, y_sample)
    importances = pd.Series(ab_imp.feature_importances_, index=X.columns)
    feats_B     = importances.nlargest(100).index.tolist()
    feats_C     = [f for f, keep in zip(X.columns,
                   SelectFromModel(ab_imp, threshold='median', prefit=True).get_support()) if keep]

    rfe = RFE(DecisionTreeClassifier(max_depth=3, random_state=SEED),
              n_features_to_select=80, step=20)
    rfe.fit(X_sample, y_sample)
    feats_D = X.columns[rfe.support_].tolist()

    return feats_A, feats_B, feats_C, feats_D

if not run_exists('AdaBoost_Feature_Selection'):
    with mlflow.start_run(run_name='AdaBoost_Feature_Selection'):
        feats_A, feats_B, feats_C, feats_D = _compute_all_feature_sets(X, X_sample, y_sample)

        score_A = cross_val_score(AdaBoostClassifier(**BASE_PARAMS),
                                  X_sample[feats_A], y_sample,
                                  cv=SKF_FAST, scoring='roc_auc').mean()
        with mlflow.start_run(run_name='AB_FS_A_corr', nested=True):
            m_A = AdaBoostClassifier(**BASE_PARAMS)
            m_A.fit(X_sample[feats_A], y_sample)
            mlflow.log_metric('cv_auc', score_A)
            mlflow.log_param('n_features', len(feats_A))
            mlflow.sklearn.log_model(m_A, artifact_path='model',
                                     registered_model_name='AB_FS_A_corr')
        print(f'A done: {score_A:.5f}')

        score_B = cross_val_score(AdaBoostClassifier(**BASE_PARAMS),
                                  X_sample[feats_B], y_sample,
                                  cv=SKF_FAST, scoring='roc_auc').mean()
        with mlflow.start_run(run_name='AB_FS_B_top100', nested=True):
            m_B = AdaBoostClassifier(**BASE_PARAMS)
            m_B.fit(X_sample[feats_B], y_sample)
            mlflow.log_metric('cv_auc', score_B)
            mlflow.log_param('n_features', len(feats_B))
            mlflow.sklearn.log_model(m_B, artifact_path='model',
                                     registered_model_name='AB_FS_B_top100')
        print(f'B done: {score_B:.5f}')

        score_C = cross_val_score(AdaBoostClassifier(**BASE_PARAMS),
                                  X_sample[feats_C], y_sample,
                                  cv=SKF_FAST, scoring='roc_auc').mean()
        with mlflow.start_run(run_name='AB_FS_C_sfm', nested=True):
            m_C = AdaBoostClassifier(**BASE_PARAMS)
            m_C.fit(X_sample[feats_C], y_sample)
            mlflow.log_metric('cv_auc', score_C)
            mlflow.log_param('n_features', len(feats_C))
            mlflow.sklearn.log_model(m_C, artifact_path='model',
                                     registered_model_name='AB_FS_C_sfm')
        print(f'C done: {score_C:.5f}')

        score_D = cross_val_score(AdaBoostClassifier(**BASE_PARAMS),
                                  X_sample[feats_D], y_sample,
                                  cv=SKF_FAST, scoring='roc_auc').mean()
        with mlflow.start_run(run_name='AB_FS_D_rfe', nested=True):
            m_D = AdaBoostClassifier(**BASE_PARAMS)
            m_D.fit(X_sample[feats_D], y_sample)
            mlflow.log_metric('cv_auc', score_D)
            mlflow.log_param('n_features', len(feats_D))
            mlflow.sklearn.log_model(m_D, artifact_path='model',
                                     registered_model_name='AB_FS_D_rfe')
        print(f'D done: {score_D:.5f}')

        BEST_FS_NAME, BEST_FS_SCORE, SELECTED_FEATURES = max(
            [('A_corr',   score_A, feats_A),
             ('B_top100', score_B, feats_B),
             ('C_sfm',    score_C, feats_C),
             ('D_rfe',    score_D, feats_D)],
            key=lambda x: x[1]
        )

        mlflow.log_metrics({'score_A': score_A, 'score_B': score_B,
                            'score_C': score_C, 'score_D': score_D,
                            'best_score': BEST_FS_SCORE})
        mlflow.log_param('best_approach', BEST_FS_NAME)
        print(f'Best: {BEST_FS_NAME} — {BEST_FS_SCORE:.5f} ({len(SELECTED_FEATURES)} features)')

else:
    print('AdaBoost_Feature_Selection already logged, recovering from MLflow...')
    client = MlflowClient()

    # Search all experiments in case active experiment name drifted
    all_exp_ids = [e.experiment_id for e in client.search_experiments()]
    runs = client.search_runs(
        experiment_ids=all_exp_ids,
        filter_string="tags.mlflow.runName = 'AdaBoost_Feature_Selection'"
    )

    if not runs:
        raise ValueError("No run named 'AdaBoost_Feature_Selection' found. Delete the experiment and rerun.")

    run = runs[0]
    print(f"Run id: {run.info.run_id} | Params logged: {list(run.data.params.keys())}")

    BEST_FS_NAME = run.data.params.get('best_approach')

    # If param is missing (run crashed before log_param), recompute scores to find best
    if BEST_FS_NAME is None:
        print("'best_approach' param missing — recomputing scores to determine best feature set...")
        feats_A, feats_B, feats_C, feats_D = _compute_all_feature_sets(X, X_sample, y_sample)
        score_A = cross_val_score(AdaBoostClassifier(**BASE_PARAMS), X_sample[feats_A], y_sample, cv=SKF_FAST, scoring='roc_auc').mean()
        score_B = cross_val_score(AdaBoostClassifier(**BASE_PARAMS), X_sample[feats_B], y_sample, cv=SKF_FAST, scoring='roc_auc').mean()
        score_C = cross_val_score(AdaBoostClassifier(**BASE_PARAMS), X_sample[feats_C], y_sample, cv=SKF_FAST, scoring='roc_auc').mean()
        score_D = cross_val_score(AdaBoostClassifier(**BASE_PARAMS), X_sample[feats_D], y_sample, cv=SKF_FAST, scoring='roc_auc').mean()
        BEST_FS_NAME, BEST_FS_SCORE, SELECTED_FEATURES = max(
            [('A_corr',   score_A, feats_A),
             ('B_top100', score_B, feats_B),
             ('C_sfm',    score_C, feats_C),
             ('D_rfe',    score_D, feats_D)],
            key=lambda x: x[1]
        )
        print(f"Recomputed best: {BEST_FS_NAME} ({BEST_FS_SCORE:.5f})")
    else:
        print(f'Best approach was: {BEST_FS_NAME}')
        feats_A, feats_B, feats_C, feats_D = _compute_all_feature_sets(X, X_sample, y_sample)
        SELECTED_FEATURES = {'A_corr': feats_A, 'B_top100': feats_B,
                             'C_sfm': feats_C, 'D_rfe': feats_D}[BEST_FS_NAME]

    print(f'SELECTED_FEATURES recovered: {len(SELECTED_FEATURES)} features')

## Training

In [ ]:
X_sel  = X[SELECTED_FEATURES]
X_test = X_test[SELECTED_FEATURES]

X_tr, X_val, y_tr, y_val = train_test_split(
    X_sel, y, test_size=0.2, random_state=SEED, stratify=y
)

def get_state(train_auc, val_auc):
    gap = train_auc - val_auc
    if val_auc < 0.80 and gap < 0.02:
        return 'underfitting'
    elif gap > 0.05:
        return 'overfitting'
    else:
        return 'good'

In [ ]:
def run_experiment(run_name, params, X_tr, y_tr, X_val, y_val):
    if run_exists(run_name):
        print(f'{run_name} already logged, skipping...')
        return
    with mlflow.start_run(run_name=run_name):
        est   = DecisionTreeClassifier(max_depth=params.pop('base_max_depth', 2),
                                       random_state=SEED)
        m     = AdaBoostClassifier(estimator=est, **params)
        m.fit(X_tr, y_tr)
        train_auc = roc_auc_score(y_tr, m.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
        state     = get_state(train_auc, val_auc)
        print(f'{run_name}: Train={train_auc:.5f}  Val={val_auc:.5f}  → {state}')
        mlflow.log_params(params)
        mlflow.log_metrics({'train_auc': train_auc, 'val_auc': val_auc,
                            'gap': train_auc - val_auc})
        mlflow.log_param('model_state', state)

In [ ]:
run_experiment(
    run_name = 'AdaBoost_n10_depth1',
    params   = dict(n_estimators=10, learning_rate=1.0,
                    base_max_depth=1, random_state=SEED),
    X_tr=X_tr, y_tr=y_tr, X_val=X_val, y_val=y_val
)

In [ ]:
run_experiment(
    run_name = 'AdaBoost_depth3_slowlr',
    params   = dict(n_estimators=200, learning_rate=0.05, base_max_depth=3, random_state=SEED),
    X_tr=X_tr, y_tr=y_tr, X_val=X_val, y_val=y_val
)

In [ ]:
run_experiment(
    run_name = 'AdaBoost_n100_depth2_baseline',
    params   = dict(n_estimators=100, learning_rate=0.1,
                    base_max_depth=2, random_state=SEED),
    X_tr=X_tr, y_tr=y_tr, X_val=X_val, y_val=y_val
)

In [ ]:
def objective(trial):
    base_max_depth = trial.suggest_int('base_max_depth', 1, 4)
    params = dict(
        n_estimators  = trial.suggest_int('n_estimators', 50, 500, step=50),
        learning_rate = trial.suggest_float('learning_rate', 0.01, 1.0, log=True),
        random_state  = SEED
    )
    est     = DecisionTreeClassifier(max_depth=base_max_depth, random_state=SEED)
    m       = AdaBoostClassifier(estimator=est, **params)
    m.fit(X_tr, y_tr)
    val_auc = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
    with mlflow.start_run(run_name=f'AB_trial_{trial.number}', nested=True):
        mlflow.log_params({**params, 'base_max_depth': base_max_depth})
        mlflow.log_metric('val_auc', val_auc)
    return val_auc

if not run_exists('AdaBoost_HyperparamSearch'):
    with mlflow.start_run(run_name='AdaBoost_HyperparamSearch'):
        study = optuna.create_study(direction='maximize',
                                    sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(objective, n_trials=20, show_progress_bar=True)
        best_params = study.best_params
        best_score  = study.best_value
        print(f'Best Val AUC: {best_score:.5f}')
        mlflow.log_params({f'best_{k}': v for k, v in best_params.items()})
        mlflow.log_metric('best_val_auc', best_score)
else:
    print('AdaBoost_HyperparamSearch already logged, recovering best_params...')
    run = get_run('AdaBoost_HyperparamSearch')
    best_params = {
        'n_estimators'  : int(run.data.params['best_n_estimators']),
        'learning_rate' : float(run.data.params['best_learning_rate']),
        'base_max_depth': int(run.data.params['best_base_max_depth']),
    }
    print('best_params recovered:', best_params)

In [ ]:
if not run_exists('AdaBoost_Final_Model'):
    with mlflow.start_run(run_name='AdaBoost_Final_Model'):
        FINAL_PARAMS = dict(
            estimator     = DecisionTreeClassifier(max_depth=best_params['base_max_depth'], random_state=SEED),
            n_estimators  = best_params['n_estimators'],
            learning_rate = best_params['learning_rate'],
            random_state  = SEED
        )

        oof = np.zeros(len(y))
        for fold, (tr_idx, val_idx) in enumerate(SKF.split(X_sel, y), 1):
            m = AdaBoostClassifier(**FINAL_PARAMS)
            m.fit(X_sel.iloc[tr_idx], y.iloc[tr_idx])
            oof[val_idx] = m.predict_proba(X_sel.iloc[val_idx])[:, 1]
            print(f'Fold {fold} done')

        final_auc = roc_auc_score(y, oof)
        print(f'Final OOF AUC: {final_auc:.5f}')

        preprocessor.feature_names_ = SELECTED_FEATURES
        final_clf = AdaBoostClassifier(**FINAL_PARAMS)
        final_clf.fit(X_sel, y)

        fraud_pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier',   final_clf)
        ])

        mlflow.log_metric('final_oof_auc', final_auc)
        mlflow.sklearn.log_model(
            fraud_pipeline,
            artifact_path='adaboost_pipeline',
            registered_model_name='AdaBoost_Fraud_Pipeline'
        )
else:
    print('AdaBoost_Final_Model already logged, skipping...')